# LightGBM — Walmart Store Sales Forecasting

In [1]:
!pip install kaggle wandb onnx -Uq
from google.colab import drive
drive.mount('/content/drive')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.5/111.5 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 46.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 243.8/243.8 kB 11.1 MB/s eta 0:00:00
Mounted at /content/drive


In [2]:
! mkdir ~/.kaggle

In [3]:
!cp /content/drive/MyDrive/MLFinalAssignment/kaggle.json ~/.kaggle/kaggle.json

In [4]:
! chmod 600 ~/.kaggle/kaggle.json

In [5]:
! kaggle competitions download walmart-recruiting-store-sales-forecasting

100% 2.70M/2.70M [00:01<00:00, 2.26MB/s]



In [6]:
! unzip -o walmart-recruiting-store-sales-forecasting

Archive:  walmart-recruiting-store-sales-forecasting.zip
  inflating: features.csv.zip        
  inflating: sampleSubmission.csv.zip  
  inflating: stores.csv              
  inflating: test.csv.zip            
  inflating: train.csv.zip           


In [ ]:
import os, pathlib, zipfile, shutil

ROOT = pathlib.Path("/content/drive/MyDrive/MLFinalAssignment")

for name in ("train", "test", "features", "sampleSubmission"):
    if not (ROOT / f"{name}.csv").exists():
        with zipfile.ZipFile(f"/content/{name}.csv.zip") as z:
            z.extractall(ROOT)
if not (ROOT / "stores.csv").exists():
    shutil.copy("/content/stores.csv", ROOT)

os.environ["WALMART_ROOT"] = str(ROOT)

print(sorted(p.name for p in ROOT.iterdir() if p.suffix == ".csv"))
assert (ROOT / "src" / "walmart_prep.py").exists(), "copy src/ into the Drive folder first"


['features.csv', 'sampleSubmission.csv', 'stores.csv', 'test.csv', 'train.csv']


In [ ]:
import importlib.util
import os
import pathlib
import subprocess
import sys

IN_COLAB = importlib.util.find_spec("google.colab") is not None

if IN_COLAB:
    missing = [p for p in ("lightgbm", "mlflow", "dagshub")
               if importlib.util.find_spec(p) is None]
    if missing:
        print("installing", *missing)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
    if not pathlib.Path("/content/drive").exists():
        from google.colab import drive
        drive.mount("/content/drive")


def find_root() -> pathlib.Path:
    """Locate the repo: it must hold train.csv and src/walmart_prep.py."""
    candidates = []
    if os.environ.get("WALMART_ROOT"):
        candidates.append(pathlib.Path(os.environ["WALMART_ROOT"]))
    candidates += [pathlib.Path.cwd(), pathlib.Path.cwd().parent]
    if IN_COLAB:
        drive_root = pathlib.Path("/content/drive/MyDrive")
        candidates += [drive_root / "MLFinalProject", pathlib.Path("/content/MLFinalProject")]
        if drive_root.exists():
            candidates += sorted(p for p in drive_root.glob("*") if (p / "train.csv").exists())
    for c in candidates:
        if (c / "train.csv").exists() and (c / "src" / "walmart_prep.py").exists():
            return c.resolve()
    raise FileNotFoundError(
        "Could not find the project. Set WALMART_ROOT to the folder containing "
        "train.csv and src/walmart_prep.py, then re-run this cell.\n"
        f"Looked in: {[str(c) for c in candidates]}")


ROOT = find_root()
sys.path.insert(0, str(ROOT / "src"))
for sub in ("docs", "submissions"):
    (ROOT / sub).mkdir(exist_ok=True)

print(f"IN_COLAB={IN_COLAB}\nROOT={ROOT}")

installing mlflow dagshub
IN_COLAB=True
ROOT=/content/drive/MyDrive/MLFinalAssignment


In [ ]:
import dagshub
import mlflow

dagshub.init(repo_owner='smama23', repo_name='MLFinalProject', mlflow=True)

EXPERIMENT_NAME = 'LightGBM_Training'
REGISTERED_MODEL_NAME = 'WalmartSalesForecast'

mlflow.set_experiment(EXPERIMENT_NAME)

if mlflow.active_run() is not None:
    mlflow.end_run()

print('Tracking URI:', mlflow.get_tracking_uri())
print('Experiment  :', EXPERIMENT_NAME)

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=9fa293c1-5dd6-42b9-ae5d-e85011b37833&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=15dd4958747838936f70ef6606d8908c0544aeb8c3b02940dd6b4191121f66e8




Accessing as smama23

Initialized MLflow to track repo "smama23/MLFinalProject"

Repository smama23/MLFinalProject initialized!

Tracking URI: https://dagshub.com/smama23/MLFinalProject.mlflow
Experiment  : LightGBM_Training


In [ ]:
import time

import numpy as np
import pandas as pd

import lightgbm as lgb
import mlflow
import mlflow.sklearn
from sklearn.pipeline import Pipeline

from walmart_prep import (
    FOLDS, HORIZON, MARKDOWN_COLS, MIN_SAFE_LAG,
    SeasonalResidualRegressor, WalmartFeatureBuilder,
    december_shape_report, load_raw, make_submission,
    score_fold, seasonal_naive, wmae_weights,
)

EXPERIMENT = EXPERIMENT_NAME
print("tracking:", mlflow.get_tracking_uri())

train, test, features, stores = load_raw(str(ROOT))
SEED = 0

print(f"train {train.shape}   test {test.shape}")
print(f"train {train.Date.min().date()} .. {train.Date.max().date()}")
print(f"test  {test.Date.min().date()} .. {test.Date.max().date()}")
print(f"horizon = {HORIZON} weeks  ->  minimum safe lag = {MIN_SAFE_LAG} weeks")

tracking: https://dagshub.com/smama23/MLFinalProject.mlflow
train (421570, 5)   test (115064, 4)
train 2010-02-05 .. 2012-10-26
test  2012-11-02 .. 2013-07-26
horizon = 39 weeks  ->  minimum safe lag = 39 weeks


In [ ]:
DEVICE_PARAMS = {"force_col_wise": True}
DEVICE = "cpu"

print(f"device={DEVICE}  cores={os.cpu_count()}")
DEVICE_PARAMS

device=cpu  cores=2


{'force_col_wise': True}

## `LightGBM_Cleaning`


In [12]:
with mlflow.start_run(run_name="LightGBM_Cleaning"):
    mlflow.log_params({
        "internal_gaps": "fill 0 (94.5% of sales next to a gap are < $500)",
        "markdowns": "fill 0 + markdown_era flag (two different missingness mechanisms)",
        "cpi_unemployment": "ffill per store, then deviation from the store's train mean",
        "negative_sales": "kept (0.305% of rows, genuine returns/adjustments)",
        "target_transform": "none (WMAE is L1; log1p would reshape the loss)",
        "store_dept_encoding": "numeric (categorical split search overfit 81 departments)",
    })

    counts = train.groupby(["Store", "Dept"]).size()
    span = train.groupby(["Store", "Dept"]).Date.agg(["min", "max"])
    span["weeks"] = (span["max"] - span["min"]).dt.days // 7 + 1
    gap_cells = int((span["weeks"] - counts).clip(lower=0).sum())

    tr_pairs = set(map(tuple, train[["Store", "Dept"]].drop_duplicates().values))
    te_pairs = set(map(tuple, test[["Store", "Dept"]].drop_duplicates().values))
    cold = sorted(te_pairs - tr_pairs)
    cold_rows = int(test.merge(pd.DataFrame(cold, columns=["Store", "Dept"]),
                               on=["Store", "Dept"]).shape[0])

    stats = {
        "train_rows": len(train),
        "test_rows": len(test),
        "train_pairs": len(tr_pairs),
        "test_pairs": len(te_pairs),
        "cold_start_pairs": len(cold),
        "cold_start_rows": cold_rows,
        "negative_sales_frac": float((train.Weekly_Sales < 0).mean()),
        "internal_gap_cells": gap_cells,
        "markdown_era_share_train": float((train.Date >= pd.Timestamp("2011-11-11")).mean()),
        "markdown_era_share_test": 1.0,
        "holiday_weight_share_test": float(
            5 * test.IsHoliday.sum() / wmae_weights(test.IsHoliday).sum()),
    }
    mlflow.log_metrics(stats)

    na = features.isna().sum().to_frame("n_missing")
    na["pct"] = (100 * features.isna().mean()).round(2)
    path = ROOT / "docs" / "features_missingness.csv"
    na.to_csv(path)
    mlflow.log_artifact(str(path))

pd.Series(stats).to_frame("value")

🏃 View run LightGBM_Cleaning at: https://dagshub.com/smama23/MLFinalProject.mlflow/#/experiments/0/runs/2927f06978d446f1a1491cbdb86a1589
🧪 View experiment at: https://dagshub.com/smama23/MLFinalProject.mlflow/#/experiments/0


,value
train_rows,421570.000000
test_rows,115064.000000
train_pairs,3331.000000
test_pairs,3169.000000
cold_start_pairs,11.000000
cold_start_rows,36.000000
negative_sales_frac,0.003048
internal_gap_cells,27667.000000
markdown_era_share_train,0.359210
markdown_era_share_test,1.000000


## `LightGBM_Baseline`

In [13]:
baseline_scores = {}
with mlflow.start_run(run_name="LightGBM_Baseline"):
    mlflow.log_param("model", "seasonal naive: lag-52, fallback pair median, then global median")
    for fold in FOLDS:
        tr = train[train.Date <= fold.cut]
        va = train[(train.Date >= fold.val_start) & (train.Date <= fold.val_end)]
        s = score_fold(va.Weekly_Sales, seasonal_naive(tr, va), va)
        baseline_scores[fold.name] = s
        mlflow.log_metric(f"wmae_{fold.name}", s["wmae"])
        for k, v in s.items():
            if k.startswith("mae_"):
                mlflow.log_metric(f"{k}_{fold.name}", v)

BASELINE = {k: v["wmae"] for k, v in baseline_scores.items()}
pd.DataFrame(baseline_scores).round(1)

🏃 View run LightGBM_Baseline at: https://dagshub.com/smama23/MLFinalProject.mlflow/#/experiments/0/runs/c2307e9183ae4ecd9168e509bd11683d
🧪 View experiment at: https://dagshub.com/smama23/MLFinalProject.mlflow/#/experiments/0


,mirror,recent,early
wmae,2037.8,1807.2,2018.6
mae,1949.9,1805.5,1955.9
mae_holiday,2320.2,1815.7,2170.0
mae_nonholiday,1918.6,1804.9,1931.2
bias,-381.1,-375.3,-341.3
mae_thanksgiving,2387.6,NaN,2399.5
mae_christmas,2711.7,NaN,2721.0
mae_superbowl,1860.7,1832.5,1874.9
mae_laborday,NaN,1798.6,1675.8


## `LightGBM_CV`

In [ ]:
DEFAULT_PARAMS = dict(
    objective="l1",
    n_estimators=900,
    learning_rate=0.05,
    num_leaves=63,
    min_child_samples=40,
    subsample=1.0,
    subsample_freq=0,
    colsample_bytree=1.0,
    reg_lambda=1.0,
    max_bin=255,
    n_jobs=-1,
    random_state=SEED,
    verbose=-1,
    **DEVICE_PARAMS,
)

FOLD_DATA = {}
for fold in FOLDS:
    t0 = time.time()
    tr = train[train.Date <= fold.cut]
    va = train[(train.Date >= fold.val_start) & (train.Date <= fold.val_end)]
    fb = WalmartFeatureBuilder(features, stores)
    Xtr = fb.fit_transform(tr.drop(columns=["Weekly_Sales"]), tr.Weekly_Sales)
    Xva = fb.transform(va.drop(columns=["Weekly_Sales"]))
    FOLD_DATA[fold.name] = dict(fb=fb, Xtr=Xtr, Xva=Xva, va=va,
                                ytr=tr.Weekly_Sales.to_numpy(),
                                w=wmae_weights(tr.IsHoliday))
    print(f"{fold.name:7s} {tr.Date.nunique():3d} train weeks | {Xtr.shape[1]:2d} features "
          f"| {time.time() - t0:4.1f}s | xmas seasons={fb.n_xmas_seasons_} "
          f"| dropped: {fb.dropped_columns_ or 'none'}")


def fit_fold(fold_name, params, drop=()):
    """Fit the residual learner on a cached fold. Returns (model, scores)."""
    d = FOLD_DATA[fold_name]
    keep = [c for c in d["Xtr"].columns if c not in drop]
    model = SeasonalResidualRegressor(lgb.LGBMRegressor(**params))
    model.fit(d["Xtr"][keep], d["ytr"], sample_weight=d["w"])
    pred = model.predict(d["Xva"][keep])
    return model, score_fold(d["va"].Weekly_Sales, pred, d["va"])

mirror   91 train weeks | 42 features |  5.2s | xmas seasons=1 | dropped: ['MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5', 'lag_104', 'markdown_era', 'md_any', 'md_total', 'yoy_trend']
recent  104 train weeks | 51 features |  7.9s | xmas seasons=2 | dropped: ['lag_104']
early    78 train weeks | 42 features |  4.4s | xmas seasons=1 | dropped: ['MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5', 'lag_104', 'markdown_era', 'md_any', 'md_total', 'yoy_trend']


In [15]:
cv_scores = {}
with mlflow.start_run(run_name="LightGBM_CV"):
    mlflow.log_params({**DEFAULT_PARAMS, "device": DEVICE,
                       "target": "residual (y - seasonal baseline)",
                       "sample_weight": "5 on holiday weeks"})
    for fold in FOLDS:
        _, s = fit_fold(fold.name, DEFAULT_PARAMS)
        cv_scores[fold.name] = s
        mlflow.log_metric(f"wmae_{fold.name}", s["wmae"])
        for k, v in s.items():
            if k.startswith("mae_"):
                mlflow.log_metric(f"{k}_{fold.name}", v)
        gain = 100 * (1 - s["wmae"] / BASELINE[fold.name])
        mlflow.log_metric(f"gain_vs_naive_{fold.name}", gain)
        print(f"{fold.name:7s} naive {BASELINE[fold.name]:7.1f} -> LGBM {s['wmae']:7.1f}  ({gain:+.1f}%)")
    mlflow.log_metric("wmae_mean", float(np.mean([s["wmae"] for s in cv_scores.values()])))

REF = {k: v["wmae"] for k, v in cv_scores.items()}
pd.DataFrame(cv_scores).round(1)

mirror  naive  2037.8 -> LGBM  1951.4  (+4.2%)
recent  naive  1807.2 -> LGBM  1635.8  (+9.5%)
early   naive  2018.6 -> LGBM  2046.4  (-1.4%)
🏃 View run LightGBM_CV at: https://dagshub.com/smama23/MLFinalProject.mlflow/#/experiments/0/runs/8c3d47ef3c40442c8edfcb5acd52a9e6
🧪 View experiment at: https://dagshub.com/smama23/MLFinalProject.mlflow/#/experiments/0


,mirror,recent,early
wmae,1951.4,1635.8,2046.4
mae,1837.3,1644.3,1863.4
mae_holiday,2317.3,1594.3,2488.7
mae_nonholiday,1796.8,1647.0,1791.3
bias,55.6,-178.7,-141.2
mae_thanksgiving,2224.0,NaN,2488.1
mae_christmas,3109.9,NaN,4145.5
mae_superbowl,1618.2,1562.8,1700.9
mae_laborday,NaN,1626.2,1606.6


## `LightGBM_Feature_Selection`

In [ ]:
BLOCKS = {
    "drop_time_index": ["t", "year"],
    "drop_markdowns": MARKDOWN_COLS + ["md_total", "md_any", "markdown_era"],
    "drop_exogenous": ["Temperature", "Fuel_Price", "CPI_dev", "Unemployment_dev"],
    "drop_xmas_aligned_lag": ["xmas_aligned_lag"],
    "drop_shared_strength": ["store_lag_52", "dept_lag_52", "pair_share_of_store",
                             "Size_per_dept_sale"],
}

rows = [{"ablation": "keep everything", **{f.name: REF[f.name] for f in FOLDS}}]
with mlflow.start_run(run_name="LightGBM_Feature_Selection"):
    mlflow.log_metrics({f"wmae_{k}_reference": v for k, v in REF.items()})

    m, _ = fit_fold("mirror", DEFAULT_PARAMS)
    booster = m.estimator_.booster_
    imp = (pd.Series(booster.feature_importance("gain"), index=booster.feature_name())
             .sort_values(ascending=False))
    path = ROOT / "docs" / "lightgbm_gain_importance.csv"
    imp.to_frame("gain").to_csv(path)
    mlflow.log_artifact(str(path))
    print("top 12 features by gain:\n" + imp.head(12).round(0).to_string() + "\n")

    for tag, cols in BLOCKS.items():
        with mlflow.start_run(run_name=f"LightGBM_Feature_Selection__{tag}", nested=True):
            mlflow.log_param("dropped_columns", ", ".join(cols))
            row = {"ablation": tag}
            for fold in FOLDS:
                present = [c for c in cols if c in FOLD_DATA[fold.name]["Xtr"].columns]
                if not present:
                    row[fold.name] = np.nan
                    continue
                _, s = fit_fold(fold.name, DEFAULT_PARAMS, drop=present)
                row[fold.name] = s["wmae"]
                mlflow.log_metric(f"wmae_{fold.name}", s["wmae"])
                mlflow.log_metric(f"delta_{fold.name}", s["wmae"] - REF[fold.name])
            rows.append(row)
            print(f"{tag:24s} " + "  ".join(
                f"{f.name}={row[f.name]:7.1f}" if np.isfinite(row.get(f.name, np.nan))
                else f"{f.name}=    n/a" for f in FOLDS))

ablation = pd.DataFrame(rows).set_index("ablation")
delta = ablation.subtract(pd.Series(REF), axis=1).drop(index="keep everything")
print("\nWMAE change vs keeping everything (positive = the block was helping):")
delta.round(1)

top 12 features by gain:
Dept                  1904726.0
t                     1390288.0
lag_45                 480523.0
Size                   370795.0
Store                  299321.0
roll_mean_lag39_51     191291.0
roll_std_lag52_w5      164383.0
lag_39                 158670.0
roll_std_lag39_51      137540.0
lag_52                 131607.0
lag_53                 125989.0
dept_lag_52            124398.0

drop_time_index          mirror= 2019.8  recent= 1644.5  early= 2325.9
🏃 View run LightGBM_Feature_Selection__drop_time_index at: https://dagshub.com/smama23/MLFinalProject.mlflow/#/experiments/0/runs/119f0bcbe2c24cdfaae835dbd694fb50
🧪 View experiment at: https://dagshub.com/smama23/MLFinalProject.mlflow/#/experiments/0
drop_markdowns           mirror=    n/a  recent= 1642.8  early=    n/a
🏃 View run LightGBM_Feature_Selection__drop_markdowns at: https://dagshub.com/smama23/MLFinalProject.mlflow/#/experiments/0/runs/a96a12102cf9489eb5cc81ef5e255774
🧪 View experiment at: https://dagsh

,mirror,recent,early
ablation,,,
drop_time_index,68.4,8.8,279.5
drop_markdowns,NaN,7.1,NaN
drop_exogenous,26.7,21.2,59.6
drop_xmas_aligned_lag,-68.6,6.9,-110.9
drop_shared_strength,5.4,5.8,-37.3


## `LightGBM_Tuning`

In [ ]:
rng = np.random.default_rng(SEED)
GRID = {
    "learning_rate": [0.03, 0.05, 0.08],
    "num_leaves": [31, 63, 127, 255],
    "min_child_samples": [20, 40, 80, 120],
    "colsample_bytree": [0.6, 0.8, 1.0],
    "subsample": [0.7, 0.9, 1.0],
    "reg_lambda": [0.0, 1.0, 5.0],
    "n_estimators": [600, 900, 1200, 1500],
    "max_bin": [63, 127, 255],
}
N_TRIALS = 12


def sample_params():
    p = dict(DEFAULT_PARAMS)
    for k, v in GRID.items():
        p[k] = type(v[0])(rng.choice(v))
    p["subsample_freq"] = 0 if p["subsample"] >= 1.0 else 1
    return p


candidates = [("default", dict(DEFAULT_PARAMS))]
candidates += [(f"trial_{i:02d}", sample_params()) for i in range(N_TRIALS)]

trials = []
with mlflow.start_run(run_name="LightGBM_Tuning"):
    mlflow.log_params({"n_trials": N_TRIALS, "search": "random + incumbent",
                       "selection_fold": "mirror", "device": DEVICE})

    for name, params in candidates:
        t0 = time.time()
        if name == "default":
            s = cv_scores["mirror"]
        else:
            _, s = fit_fold("mirror", params)
        with mlflow.start_run(run_name=f"LightGBM_Tuning__{name}", nested=True):
            mlflow.log_params(params)
            mlflow.log_metric("wmae_mirror", s["wmae"])
            if "mae_christmas" in s:
                mlflow.log_metric("mae_christmas_mirror", s["mae_christmas"])
            mlflow.log_metric("fit_seconds", time.time() - t0)
        trials.append({"name": name, "params": params, "mirror": s["wmae"]})
        print(f"{name:12s} wmae_mirror={s['wmae']:7.1f}  ({time.time() - t0:4.0f}s)  "
              f"lr={params['learning_rate']} leaves={params['num_leaves']} "
              f"n={params['n_estimators']} sub={params['subsample']} col={params['colsample_bytree']}")

    top = sorted(trials, key=lambda r: r["mirror"])[:3]
    print("\nconfirming the top 3 on every fold...")
    for r in top:
        for fold in FOLDS:
            if fold.name == "mirror":
                continue
            elif r["name"] == "default":
                r[fold.name] = REF[fold.name]
            else:
                r[fold.name] = fit_fold(fold.name, r["params"])[1]["wmae"]
        r["mean"] = float(np.mean([r[f.name] for f in FOLDS]))
        r["beats_naive"] = all(r[f.name] < BASELINE[f.name] for f in FOLDS)
        print(f"  {r['name']:12s} " + "  ".join(f"{f.name}={r[f.name]:7.1f}" for f in FOLDS)
              + f"  mean={r['mean']:7.1f}  beats_naive={r['beats_naive']}")

    eligible = [r for r in top if r["beats_naive"]] or top
    best = min(eligible, key=lambda r: r["mean"])
    BEST_PARAMS = best["params"]
    mlflow.log_params({f"best_{k}": v for k, v in BEST_PARAMS.items()})
    mlflow.log_metrics({f"best_wmae_{f.name}": best[f.name] for f in FOLDS})
    mlflow.log_metric("best_wmae_mean", best["mean"])
    mlflow.log_param("best_candidate", best["name"])

print(f"\nwinner: {best['name']}   mean WMAE {best['mean']:.1f}")
print(f"default mean WMAE {np.mean(list(REF.values())):.1f}")
BEST_PARAMS

🏃 View run LightGBM_Tuning__default at: https://dagshub.com/smama23/MLFinalProject.mlflow/#/experiments/0/runs/d8d1234230be4b65b2f4ac362e52f49b
🧪 View experiment at: https://dagshub.com/smama23/MLFinalProject.mlflow/#/experiments/0
default      wmae_mirror= 1951.4  (   2s)  lr=0.05 leaves=63 n=900 sub=1.0 col=1.0
🏃 View run LightGBM_Tuning__trial_00 at: https://dagshub.com/smama23/MLFinalProject.mlflow/#/experiments/0/runs/b7c07ec44eeb4db08051845ec2942a9f
🧪 View experiment at: https://dagshub.com/smama23/MLFinalProject.mlflow/#/experiments/0
trial_00     wmae_mirror= 1942.6  (  73s)  lr=0.08 leaves=127 n=600 sub=0.7 col=0.6
🏃 View run LightGBM_Tuning__trial_01 at: https://dagshub.com/smama23/MLFinalProject.mlflow/#/experiments/0/runs/1619338be7254b73bf71f32844cc4dc0
🧪 View experiment at: https://dagshub.com/smama23/MLFinalProject.mlflow/#/experiments/0
trial_01     wmae_mirror= 1913.2  ( 290s)  lr=0.03 leaves=255 n=1500 sub=0.9 col=1.0
🏃 View run LightGBM_Tuning__trial_02 at: https://d

{'objective': 'l1',
 'n_estimators': 900,
 'learning_rate': 0.03,
 'num_leaves': 255,
 'min_child_samples': 20,
 'subsample': 0.7,
 'subsample_freq': 1,
 'colsample_bytree': 0.8,
 'reg_lambda': 0.0,
 'max_bin': 127,
 'n_jobs': -1,
 'random_state': 0,
 'verbose': -1,
 'force_col_wise': True}

## `LightGBM_Final`

In [ ]:
with mlflow.start_run(run_name="LightGBM_Final") as final_run:
    mlflow.log_params({**BEST_PARAMS, "trained_on": "all 143 weeks",
                       "target": "residual (y - seasonal baseline)",
                       "selected_candidate": best["name"], "device": DEVICE})
    mlflow.log_metrics({f"cv_wmae_{f.name}": best[f.name] for f in FOLDS})
    mlflow.log_metric("cv_wmae_mean", best["mean"])

    pipe = Pipeline([
        ("features", WalmartFeatureBuilder(features, stores)),
        ("model", SeasonalResidualRegressor(lgb.LGBMRegressor(**BEST_PARAMS))),
    ])
    pipe.fit(train.drop(columns=["Weekly_Sales"]), train.Weekly_Sales,
             model__sample_weight=wmae_weights(train.IsHoliday))

    y_pred = pipe.predict(test)
    print(f"predictions: n={len(y_pred)}  mean={y_pred.mean():.1f}  "
          f"min={y_pred.min():.1f}  max={y_pred.max():.1f}")

    fb = pipe.named_steps["features"]
    frame, verdict = december_shape_report(train, test, y_pred, fb.xmas_profile_)
    print(f"\nprofile fitted on {fb.n_xmas_seasons_} December season(s)")
    print(frame.to_string(index=False, float_format=lambda v: f"{v:8.3f}"))
    print(f"\npredicted peak {verdict['peak_predicted']} | implied peak {verdict['peak_implied']}")
    print("GATE:", "PASS" if verdict["passed"] else "REVIEW -- " + "; ".join(verdict["problems"]))

    gate_path = ROOT / "docs" / "lightgbm_december_gate.csv"
    frame.to_csv(gate_path, index=False)
    mlflow.log_artifact(str(gate_path))
    mlflow.log_metric("december_xmas_gap_pct", verdict["xmas_gap_pct"])
    mlflow.log_metric("december_gate_passed", int(verdict["passed"]))
    mlflow.log_param("december_peak_predicted", verdict["peak_predicted"])

    logged_model = mlflow.sklearn.log_model(
        pipe, name="pipeline",
        code_paths=[str(ROOT / "src" / "walmart_prep.py")],
        input_example=test.head(3),
        registered_model_name=REGISTERED_MODEL_NAME,
        serialization_format="cloudpickle",
    )
    MODEL_URI = logged_model.model_uri

    sub = make_submission(test, y_pred, ROOT / "submissions" / "lightgbm.csv")
    mlflow.log_artifact(str(ROOT / "submissions" / "lightgbm.csv"))
    FINAL_RUN_ID = final_run.info.run_id

print(f"\nrun_id    = {FINAL_RUN_ID}")
print(f"model_uri = {MODEL_URI}")
sub.head()

predictions: n=115064  mean=16664.2  min=-2927.5  max=634556.3

profile fitted on 2 December season(s)
      week  k_end  predicted  implied  gap_pct
2012-11-30    -25      0.987    0.969    1.816
2012-12-07    -18      1.108    1.109   -0.111
2012-12-14    -11      1.177    1.208   -2.586
2012-12-21     -4      1.452    1.489   -2.496
2012-12-28      3      1.141    1.243   -8.159
2013-01-04     10      0.952    0.919    3.609

predicted peak 2012-12-21 | implied peak 2012-12-21
GATE: PASS


2026/07/10 16:25:06 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
/usr/local/lib/python3.12/dist-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling 

🏃 View run LightGBM_Final at: https://dagshub.com/smama23/MLFinalProject.mlflow/#/experiments/0/runs/8b17e318f144477d823b4ae071786d1e
🧪 View experiment at: https://dagshub.com/smama23/MLFinalProject.mlflow/#/experiments/0

run_id    = 8b17e318f144477d823b4ae071786d1e
model_uri = models:/m-3ab1967a6dfd469abc64ea2997dd62d4


,Id,Weekly_Sales
0,1_1_2012-11-02,37625.495865
1,1_1_2012-11-09,20553.891882
2,1_1_2012-11-16,19836.056944
3,1_1_2012-11-23,21871.944174
4,1_1_2012-11-30,25307.626955


In [ ]:
loaded = mlflow.sklearn.load_model(MODEL_URI)
reloaded_pred = loaded.predict(test)
print("max |reloaded - original| =", float(np.abs(reloaded_pred - y_pred).max()))
assert np.allclose(reloaded_pred, y_pred), "the logged Pipeline must reproduce its predictions"
print("Pipeline round-trip OK -- ready for the Model Registry")

max |reloaded - original| = 0.0
Pipeline round-trip OK -- ready for the Model Registry


In [23]:
print("experiments: https://dagshub.com/smama23/MLFinalProject/experiments")
print("registry   : https://dagshub.com/smama23/MLFinalProject/models")

runs = mlflow.search_runs(experiment_names=[EXPERIMENT])
cols = [c for c in ("tags.mlflow.runName", "metrics.wmae_mirror", "metrics.wmae_recent",
                    "metrics.wmae_early", "metrics.december_gate_passed") if c in runs.columns]
display(runs[cols])

experiments: https://dagshub.com/smama23/MLFinalProject/experiments
registry   : https://dagshub.com/smama23/MLFinalProject/models


,tags.mlflow.runName,metrics.wmae_mirror,metrics.wmae_recent,metrics.wmae_early,metrics.december_gate_passed
0,LightGBM_Final,NaN,NaN,NaN,1.0
1,LightGBM_Final,NaN,NaN,NaN,1.0
2,LightGBM_Final,NaN,NaN,NaN,1.0
3,LightGBM_Tuning__trial_11,2025.082177,NaN,NaN,NaN
4,LightGBM_Tuning__trial_10,1937.841846,NaN,NaN,NaN
5,LightGBM_Tuning__trial_09,1975.764843,NaN,NaN,NaN
6,LightGBM_Tuning__trial_08,1935.901418,NaN,NaN,NaN
7,LightGBM_Tuning__trial_07,1932.863635,NaN,NaN,NaN
8,LightGBM_Tuning__trial_06,1912.126974,NaN,NaN,NaN
9,LightGBM_Tuning__trial_05,1871.076932,NaN,NaN,NaN
